# Modelos de predicción

En este capítulo predecimos el precio. Todo se evalúa por **zonas de la ciudad**: el modelo se
prueba en zonas que no vio al entrenar, porque así se usaría para valorar una vivienda nueva y
porque el capítulo 5 mostró que el precio de una vivienda se parece mucho al de sus vecinas.

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "4")   # evita que XGBoost y scikit-learn se bloqueen en Jupyter
from eda_utils import *
import time
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, ParameterSampler
from sklearn.neighbors import BallTree
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor

## Ventas registradas al avalúo

Al revisar dónde fallaba una primera versión del modelo encontramos que muchas ventas no
registran el precio de mercado sino el avalúo catastral: el precio, llevado a pesos de 2022 (el
año del avalúo), queda a menos de 10% del avalúo. Pasa sobre todo en la fuente `igac`.

In [2]:
d = viv[viv.lat.notna() & viv.lon.notna() & viv.area.notna()].copy().reset_index(drop=True)
a_2022 = ipc_anio.loc[2022] / ipc_anio
d["razon_avaluo"] = d.precio * d.anio.map(a_2022) / d.avaluo
d["al_avaluo"] = d.razon_avaluo.between(0.9, 1.1).fillna(False).astype(bool)   # sin avalúo: no se puede saber
d["tipo_txt"] = np.where((d.condicion_predio == "NPH").fillna(False), "casa", "apartamento u otro")

(d.groupby(["tipo_txt", "fuente"])
  .agg(ventas=("precio", "size"), precio_sobre_avaluo=("razon_avaluo", "median"),
       pct_al_avaluo=("al_avaluo", "mean"))
  .assign(pct_al_avaluo=lambda t: (t.pct_al_avaluo * 100).round(1)).round(2))

ventas  precio_sobre_avaluo  pct_al_avaluo
tipo_txt           fuente                                                       
apartamento u otro georreferenciadas    1383                 1.72           1.00
                   igac                   41                 1.01          48.80
                   observatorio         8472                 1.42           1.00
casa               georreferenciadas     188                 1.65           8.00
                   igac                 2507                 1.00          57.80
                   observatorio         2868                 1.82           5.20

En las casas del IGAC el precio mediano es exactamente el avalúo (1,00) y el 58% de las ventas queda a menos de 10% de él; en el observatorio pasa con el 5% de las casas y con el 1% de los apartamentos. Esas ventas no reflejan el mercado: lo más probable es que en la escritura se haya declarado el avalúo y no lo que realmente se pagó. Las quitamos del modelo. Las ventas sin avalúo se conservan, porque en ellas no se puede saber.

Esto también afecta al capítulo 4: la razón mediana de 1,54 entre precio y avalúo incluye estas ventas, así que la brecha real del mercado es mayor.

In [3]:
print(f"Ventas registradas al avalúo: {d.al_avaluo.sum():,} de {len(d):,} ({d.al_avaluo.mean():.1%})")
d = d[~d.al_avaluo].reset_index(drop=True)
print(f"Base para modelar: {len(d):,} ventas")

Ventas registradas al avalúo: 1,735 de 15,459 (11.2%)
Base para modelar: 13,724 ventas


## Variables y protocolo de evaluación

Las variables son el área (en logaritmo), habitaciones, baños, piso, plantas, edad, estrato,
latitud, longitud, año de venta, tipo de predio y **la fuente del dato**. La fuente se incluye
porque cada una registra el precio de forma distinta, y así el modelo puede corregirlo. La
variable objetivo es el logaritmo del precio en pesos de 2025.

Dividimos la ciudad en bloques de 0,01° (unos 1,1 km de lado). Bloques elegidos al azar que suman
cerca del 20% de las ventas quedan aparte como **prueba final** y se usan una sola vez, al final.
Con el resto comparamos modelos con validación cruzada de cinco particiones, también por bloques.
El bloque de Sector Alameda del Río (unas 2.100 ventas) siempre queda en entrenamiento: si cae en
la prueba, la mitad de la prueba es un solo proyecto y se vuelve mucho más fácil que el resto de
la ciudad.

In [4]:
d["log_area"] = np.log10(d.area)
d["log_pm2"] = np.log10(d.precio_m2_real)
d["estrato_n"] = d.estrato.astype(str).map({e: i + 1 for i, e in enumerate(ESTRATOS)})
for t in ["PH_Unidad_Predial", "NPH", "PH_Matriz"]:
    d[f"tipo_{t}"] = (d.condicion_predio == t).astype(float)
for f in ["igac", "observatorio", "georreferenciadas"]:
    d[f"fuente_{f}"] = (d.fuente == f).astype(float)
d["bloque"] = (np.floor(d.lat / 0.01).astype(int).astype(str) + "_" +
               np.floor(d.lon / 0.01).astype(int).astype(str))

BASE = ["log_area", "habitaciones", "banios", "piso", "plantas", "edad", "estrato_n", "lat", "lon",
        "anio", "tipo_PH_Unidad_Predial", "tipo_NPH", "tipo_PH_Matriz",
        "fuente_igac", "fuente_observatorio", "fuente_georreferenciadas"]

bloques = d.groupby("bloque").agg(ventas=("bloque", "size"),
                                  alameda=("barrio", lambda s: (s == "SECTOR ALAMEDA DEL RIO").mean()))
candidatos = bloques[bloques.alameda < 0.5].sample(frac=1, random_state=42)
bloques_prueba = candidatos.index[candidatos.ventas.cumsum() <= 0.2 * len(d)]
es_prueba = d.bloque.isin(bloques_prueba)
ent, prueba = d[~es_prueba].reset_index(drop=True), d[es_prueba].reset_index(drop=True)
y_ent, y_prueba = ent.log_precio.to_numpy(), prueba.log_precio.to_numpy()
print(f"Entrenamiento: {len(ent):,} ventas en {ent.bloque.nunique()} bloques")
print(f"Prueba final : {len(prueba):,} ventas en {prueba.bloque.nunique()} bloques")

Entrenamiento: 11,324 ventas en 71 bloques
Prueba final : 2,400 ventas en 24 bloques


## Comparación de modelos

In [5]:
MODELOS = {
    "Regresión lineal": lambda: make_pipeline(SimpleImputer(strategy="median", add_indicator=True),
                                              LinearRegression()),
    "Random Forest": lambda: make_pipeline(SimpleImputer(strategy="median"),
                                           RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                                                 max_features=0.5, n_jobs=4, random_state=42)),
    "Gradient Boosting": lambda: HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05,
                                                               min_samples_leaf=20, random_state=42),
    "XGBoost": lambda: XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
                                    colsample_bytree=0.8, min_child_weight=3, n_jobs=4, random_state=42),
}
CV = GroupKFold(5)

def metricas(y, p):
    err = 10 ** (p - y) - 1
    return {"R²": 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "MdAPE (%)": np.median(np.abs(err)) * 100,
            "error < 20% (%)": np.mean(np.abs(err) <= 0.2) * 100}

def validar(fabrica, columnas, datos):
    """Predicciones fuera de muestra con validación cruzada por bloques."""
    pred = np.empty(len(datos))
    for tr, va in CV.split(datos, groups=datos.bloque):
        a, b = datos.iloc[tr], datos.iloc[va]
        pred[va] = fabrica().fit(a[columnas], a.log_precio).predict(b[columnas])
    return pred

sin_fuente = [c for c in BASE if not c.startswith("fuente_")]
filas = []
for nombre, fabrica in MODELOS.items():
    for etiqueta, cols in [("sin fuente", sin_fuente), ("con fuente", BASE)]:
        filas.append({"modelo": nombre, "variables": etiqueta, **metricas(y_ent, validar(fabrica, cols, ent))})
pd.DataFrame(filas).round(3)

,modelo,variables,R²,MdAPE (%),error < 20% (%)
0,Regresión lineal,sin fuente,0.60,25.82,39.80
1,Regresión lineal,con fuente,0.62,24.84,40.90
2,Random Forest,sin fuente,0.64,23.50,44.32
3,Random Forest,con fuente,0.66,23.31,43.70
4,Gradient Boosting,sin fuente,0.61,22.57,44.95
5,Gradient Boosting,con fuente,0.63,22.97,44.13
6,XGBoost,sin fuente,0.63,22.48,45.40
7,XGBoost,con fuente,0.65,23.01,44.11


Los tres modelos de árboles quedan cerca entre sí: XGBoost y Gradient Boosting tienen un error mediano de 22,5% a 23,0%, Random Forest de 23,3% a 23,5%, y la regresión lineal de 24,8% a 25,8%.

La fuente sube el R² de todos los modelos (XGBoost pasa de 0,63 a 0,65), pero en los dos modelos de boosting el error mediano empeora medio punto. Buena parte de la diferencia entre fuentes desapareció al quitar las ventas registradas al avalúo. La dejamos porque las fuentes siguen midiendo el precio de forma distinta y porque mejora el R².

## Variables del entorno

Probamos también variables que describen qué hay alrededor de cada vivienda. Salen de
OpenStreetMap, se descargaron una vez con `descargar_entorno.py` y están en
`datos/entorno_osm.csv` (© colaboradores de OpenStreetMap):

- la distancia (km) al río Magdalena, a la vía principal más cercana, a la parada de Transmetro,
  al centro comercial, a la universidad y a la clínica u hospital más cercanos;
- cuántos colegios, parques, supermercados y centros de salud hay a menos de 1 km;
- cuántas ventas hay en la misma manzana, que indica si es un edificio o conjunto grande.

In [6]:
R_TIERRA_KM = 6371
osm = pd.read_csv("datos/entorno_osm.csv")
osm = osm[(osm.categoria != "rio") | (osm.nombre == "Río Magdalena")]   # sin los caños
DISTANCIAS = {"rio": "dist_rio", "via_principal": "dist_via", "transmetro": "dist_transmetro",
              "centro_comercial": "dist_centro_comercial", "universidad": "dist_universidad",
              "salud": "dist_salud"}
CONTEOS = {"colegio": "colegios_1km", "parque": "parques_1km", "supermercado": "supermercados_1km",
           "salud": "salud_1km"}
ventas_manzana = d.manzana.value_counts()

def entorno(datos):
    xy = np.radians(datos[["lat", "lon"]].to_numpy())
    salida = pd.DataFrame(index=datos.index)
    for cat, col in DISTANCIAS.items():
        arbol = BallTree(np.radians(osm.loc[osm.categoria == cat, ["lat", "lon"]].to_numpy()), metric="haversine")
        salida[col] = arbol.query(xy, k=1)[0][:, 0] * R_TIERRA_KM
    for cat, col in CONTEOS.items():
        arbol = BallTree(np.radians(osm.loc[osm.categoria == cat, ["lat", "lon"]].to_numpy()), metric="haversine")
        salida[col] = arbol.query_radius(xy, r=1 / R_TIERRA_KM, count_only=True)
    salida["ventas_manzana"] = np.log10(datos.manzana.map(ventas_manzana))
    return salida

OSM = list(DISTANCIAS.values()) + list(CONTEOS.values()) + ["ventas_manzana"]
ent, prueba = ent.join(entorno(ent)), prueba.join(entorno(prueba))
filas = []
for nombre in ["Random Forest", "Gradient Boosting", "XGBoost"]:
    for etiqueta, cols in [("vivienda + ubicación + fuente", BASE), ("+ entorno", BASE + OSM)]:
        filas.append({"modelo": nombre, "variables": etiqueta, **metricas(y_ent, validar(MODELOS[nombre], cols, ent))})
tabla_entorno = pd.DataFrame(filas)
x = tabla_entorno[tabla_entorno.modelo == "XGBoost"].set_index("variables")["MdAPE (%)"]
USAR_ENTORNO = bool(x["+ entorno"] < x["vivienda + ubicación + fuente"])
print("¿Usar el entorno en el modelo final?", "sí" if USAR_ENTORNO else "no")
tabla_entorno.round(3)

¿Usar el entorno en el modelo final? no


,modelo,variables,R²,MdAPE (%),error < 20% (%)
0,Random Forest,vivienda + ubicación + fuente,0.66,23.31,43.70
1,Random Forest,+ entorno,0.66,23.11,43.76
2,Gradient Boosting,vivienda + ubicación + fuente,0.63,22.97,44.13
3,Gradient Boosting,+ entorno,0.63,22.90,44.88
4,XGBoost,vivienda + ubicación + fuente,0.65,23.01,44.11
5,XGBoost,+ entorno,0.64,24.60,41.73


El entorno no mejora a XGBoost: su error mediano sube de 23,0% a 24,6%. En Random Forest y Gradient Boosting cambia menos de 0,2 puntos. La distancia al río o a un centro comercial es otra forma de decir dónde está la vivienda, y eso el modelo ya lo sabe por la latitud y la longitud. Por eso el modelo final no usa estas variables.

## Ajuste de hiperparámetros

Probamos 25 combinaciones de XGBoost con la misma validación por bloques y nos quedamos con la
mejor solo si supera a la configuración inicial.

In [7]:
COLUMNAS = BASE + OSM if USAR_ENTORNO else BASE
espacio = {"learning_rate": [0.02, 0.03, 0.05, 0.08], "max_depth": [3, 4, 6, 8],
           "min_child_weight": [1, 3, 10, 30], "subsample": [0.6, 0.8, 1.0],
           "colsample_bytree": [0.5, 0.8, 1.0], "reg_lambda": [0.1, 1.0, 10.0],
           "n_estimators": [300, 600, 1000]}
inicial = dict(n_estimators=600, learning_rate=0.05, max_depth=6, subsample=0.8,
               colsample_bytree=0.8, min_child_weight=3, reg_lambda=1.0)
pruebas = []
for params in [inicial] + list(ParameterSampler(espacio, n_iter=25, random_state=42)):
    p = validar(lambda: XGBRegressor(n_jobs=4, random_state=42, **params), COLUMNAS, ent)
    pruebas.append({**params, **metricas(y_ent, p), "inicial": params is inicial})
busqueda = pd.DataFrame(pruebas).sort_values("MdAPE (%)")
mejores = {k: busqueda.iloc[0][k] for k in inicial}
mejores = {k: (int(v) if k in ("n_estimators", "max_depth", "min_child_weight") else float(v))
           for k, v in mejores.items()}
print("Configuración elegida:", mejores)
busqueda.head(5).round(3)

Configuración elegida: {'n_estimators': 600, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.6, 'colsample_bytree': 0.8, 'min_child_weight': 30, 'reg_lambda': 10.0}


,n_estimators,learning_rate,max_depth,subsample,colsample_bytree,min_child_weight,reg_lambda,R²,MdAPE (%),error < 20% (%),inicial
20,600,0.05,6,0.60,0.80,30,10.00,0.67,21.01,48.07,False
13,300,0.03,6,0.60,0.80,10,1.00,0.67,21.28,47.24,False
5,1000,0.05,3,0.80,1.00,30,10.00,0.67,21.29,47.88,False
19,600,0.08,3,0.80,1.00,30,0.10,0.67,21.35,47.38,False
9,1000,0.02,4,0.80,0.50,1,1.00,0.67,21.46,46.77,False


La mejor combinación regulariza más (`min_child_weight` = 30, `reg_lambda` = 10) y usa el 60% de las ventas en cada árbol. Baja el error mediano en validación de 23,0% a 21,0% y sube el R² a 0,67.

## Prueba final

Entrenamos el modelo elegido con todo el conjunto de entrenamiento y lo evaluamos una sola vez
en las zonas que dejamos aparte al principio. La regresión lineal queda como referencia.

In [8]:
final = XGBRegressor(n_jobs=4, random_state=42, **mejores).fit(ent[COLUMNAS], ent.log_precio)
p_final = final.predict(prueba[COLUMNAS])
p_lineal = MODELOS["Regresión lineal"]().fit(ent[BASE], ent.log_precio).predict(prueba[BASE])
pd.DataFrame({"XGBoost (modelo elegido)": metricas(y_prueba, p_final),
              "Regresión lineal": metricas(y_prueba, p_lineal)}).T.round(3)

,R²,MdAPE (%),error < 20% (%)
XGBoost (modelo elegido),0.63,25.07,40.29
Regresión lineal,0.57,26.79,39.54


In [9]:
prueba["error_pct"] = (10 ** (p_final - y_prueba) - 1) * 100
resumen = lambda g: g.agg(ventas=("error_pct", "size"), MdAPE=("error_pct", lambda e: np.median(np.abs(e))),
                          sesgo=("error_pct", "median")).round(1)
pd.concat([resumen(prueba.groupby("tipo_txt")),
           resumen(prueba.assign(loc=prueba.localidad.fillna("Sin localidad").str.title()).groupby("loc"))],
          keys=["Tipo", "Localidad"])

ventas  MdAPE  sesgo
Tipo      apartamento u otro          1541  22.20 -10.40
          casa                         859  34.10 -10.50
Localidad Metropolitana                236  33.10   5.70
          Norte - Centro Historico     105  23.10   0.40
          Riomar                       431  20.70  -9.50
          Rural                        594  25.10 -24.10
          Sin Localidad                443  20.10  -4.80
          Suroccidente                 346  33.50 -12.60
          Suroriente                   245  28.10   4.50

En las zonas de prueba XGBoost tiene un R² de 0,63 y un error mediano de 25,1%, frente a 26,8% de la regresión lineal. La prueba sale peor que la validación (21,0%), y es esperable: son solo 24 zonas y el resultado depende de cuáles cayeron ahí. Con tan pocas zonas, diferencias de dos o tres puntos pueden ser azar.

El modelo erra 22% con los apartamentos y 34% con las casas, y en la prueba tiende a subestimar: el sesgo mediano es de −10%, y en Rural llega a −24%.

### Qué variables usa el modelo

Contribución media de cada variable a la predicción (valor absoluto de SHAP, calculado de forma
exacta por XGBoost) en las ventas de prueba.

In [10]:
contrib = final.get_booster().predict(xgb.DMatrix(prueba[COLUMNAS]), pred_contribs=True)[:, :-1]
pd.Series(np.abs(contrib).mean(axis=0), index=COLUMNAS).sort_values(ascending=False).round(4).to_frame("|SHAP| medio")

,|SHAP| medio
estrato_n,0.11
log_area,0.10
lat,0.10
lon,0.03
piso,0.02
edad,0.02
fuente_observatorio,0.02
fuente_igac,0.02
banios,0.01
tipo_NPH,0.01


Estrato, área y latitud son las variables que más pesan: entre las tres suman cerca de dos tercios de la contribución total. Una vez quitadas las ventas al avalúo, la fuente pesa poco.

## Resumen y qué sigue

- El 11% de las ventas registra el avalúo en lugar del precio de mercado, sobre todo casas del IGAC. Las quitamos del modelo y conviene recalcular con ese criterio la brecha del capítulo 4.
- El mejor modelo es XGBoost ajustado: error mediano de 21,0% en validación por zonas y de 25,1% en la prueba final (R² de 0,63).
- Ni el precio de las vecinas (probado en una versión anterior de este capítulo) ni las variables del entorno mejoraron el modelo.
- El error se concentra en las casas (34%, frente a 22% en los apartamentos). Al precio de una casa le pesan mucho el área del lote, el estado y las remodelaciones, y nada de eso está en la base.

Próximos pasos:

1. Conseguir el área del terreno de las casas, que está en la capa de terrenos del catastro.
2. Recalcular la brecha precio/avalúo sin las ventas registradas al avalúo.
3. Repetir la separación de zonas de prueba varias veces para que el resultado dependa menos de qué zonas cayeron en la prueba.
4. Llevar los resultados al artículo.